descarrega do Alpha os pagos de dividendos das acções corporatívas


In [23]:
function = 'DIVIDENDS'
symbol = 'GGAL' # modificar#
apikey = 'BSFBG5DHV3CCXVMV'

In [25]:
import requests
import pandas as pd

def importa_dividendos(function: str, symbol: str, apikey: str) -> pd.DataFrame:
    url = f'https://www.alphavantage.co/query?function={function}&symbol={symbol}&apikey={apikey}'
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Garante que a requisição foi bem-sucedida
        data = response.json()
        
        if "data" in data:
            df = pd.DataFrame(data["data"])
        else:
            df = pd.DataFrame()
        
        return df
    
    except requests.exceptions.RequestException as e:
        print(f"Erro ao acessar a API: {e}")
        return pd.DataFrame()
    except ValueError:
        print("Erro ao processar a resposta da API")
        return pd.DataFrame()

# Exemplo de uso

df_dividendos = importa_dividendos(function, symbol, apikey)
print(df_dividendos)

   ex_dividend_date declaration_date record_date payment_date    amount
0        2025-06-03             None        None         None  0.263107
1        2024-08-19       2024-08-09  2024-08-19   2024-08-26  0.724786
2        2024-07-26       2024-07-16  2024-07-26   2024-08-02  0.657481
3        2024-06-21       2024-06-11  2024-06-21   2024-06-28  0.700193
4        2024-06-10       2024-05-30  2024-06-10   2024-06-17   0.29034
5        2023-09-22       2023-09-15  2023-09-25   2023-10-02  0.115391
6        2023-08-24       2023-08-15  2023-08-25   2023-09-01  0.143328
7        2023-07-28       2023-07-20  2023-07-31   2023-08-07  0.160573
8        2023-07-10       2023-07-03  2023-07-11   2023-07-17   0.17073
9        2023-05-31       2023-05-23  2023-06-01   2023-06-06  0.454093
10       2023-01-27       2023-01-20  2023-01-30   2023-02-06  0.076413
11       2022-09-30       2022-09-22  2022-10-03   2022-10-11  0.089542
12       2022-05-13       2022-05-04  2022-05-16   2022-05-16  0

In [27]:
import sqlite3
import pandas as pd

def importar_dividendos(symbol: str, df_dividendos: pd.DataFrame, db_path="sqtitulosalpha.db"):
    """
    Verifica se o símbolo existe na tabela 'tbtitulos' e obtém o id_titulos.
    Insere dividendos na tabela 'tbtitulosdividendos' apenas se id_titulos e exdividendos não existirem na tabela.
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Buscar o id_titulos correspondente ao símbolo
        cursor.execute("SELECT id_titulos FROM tbtitulos WHERE symbol = ?", (symbol,))
        result = cursor.fetchone()

        if result:
            id_titulos = result[0]
        else:
            print(f"Símbolo {symbol} não encontrado na tabela 'tbtitulos'.")
            conn.close()
            return

        # Criar a tabela 'tbtitulosdividendos' se não existir
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS tbtitulosdividendos (
            id_titulos INTEGER,
            exdividendos TEXT,
            declarationdate TEXT,
            recorddate TEXT,
            paymentdate TEXT,
            amount REAL
        )
        """)

        # Inserir apenas registros onde id_titulos e exdividendos não existam na tabela
        for _, row in df_dividendos.iterrows():
            cursor.execute("""
            SELECT COUNT(*) FROM tbtitulosdividendos 
            WHERE id_titulos = ? AND exdividendos = ?
            """, (id_titulos, row['ex_dividend_date']))
            
            if cursor.fetchone()[0] == 0:  # Se não existir, inserir
                cursor.execute("""
                INSERT INTO tbtitulosdividendos (id_titulos, exdividendos, declarationdate, recorddate, paymentdate, amount)
                VALUES (?, ?, ?, ?, ?, ?)
                """, (id_titulos, row['ex_dividend_date'], row['declaration_date'], row['record_date'], row['payment_date'], row['amount']))
                print(f"Registro {row['ex_dividend_date']} inserido.")

        conn.commit()
        conn.close()
        print("Processo concluído!")

    except sqlite3.Error as e:
        print(f"Erro ao acessar o banco de dados: {e}")

# Exemplo de uso

importar_dividendos(symbol, df_dividendos)

Registro 2024-06-21 inserido.
Registro 2024-06-10 inserido.
Registro 2023-09-22 inserido.
Registro 2023-08-24 inserido.
Registro 2023-07-28 inserido.
Registro 2023-07-10 inserido.
Registro 2023-05-31 inserido.
Registro 2023-01-27 inserido.
Registro 2022-09-30 inserido.
Registro 2022-05-13 inserido.
Registro 2021-05-14 inserido.
Registro 2020-10-09 inserido.
Registro 2019-05-15 inserido.
Registro 2018-05-11 inserido.
Registro 2017-09-26 inserido.
Registro 2017-05-10 inserido.
Registro 2016-05-19 inserido.
Registro 2015-05-26 inserido.
Registro 2014-05-22 inserido.
Registro 2013-05-15 inserido.
Registro 2012-05-15 inserido.
Registro 2011-05-11 inserido.
Registro 2004-05-26 inserido.
Registro 2001-03-27 inserido.
Processo concluído!
